# Packages

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import random as rd
from surprise import AlgoBase
from surprise.prediction_algorithms.predictions import PredictionImpossible
from sklearn.preprocessing import MinMaxScaler


from loaders import load_ratings
from loaders import load_items
from constants import Constant as C

# Explore and select content features

In [2]:
df_items = load_items()
df_ratings = load_ratings()

# Example 1 : create title_length features
df_features = df_items[C.LABEL_COL].apply(lambda x: len(x)).to_frame('n_character_title')
display(df_features.head())

# (explore here other features)

#df_rank = df_items[C.]

df_movie_date = df_items[C.LABEL_COL].str.extract(r'\((\d{4})\)').astype(float)
df_movie_date.columns = ['release_year']

# Attention : s'il manque l'année pour certains films, cela va créer des cases vides (NaN) 
# qui font planter la régression. On remplace les vides par l'année médiane :
df_movie_date['release_year'] = df_movie_date['release_year'].fillna(df_movie_date['release_year'].median())

display(df_movie_date.head())

df_movie_genres = df_items[C.GENRES_COL].str.split('|')
display(df_movie_genres)


,n_character_title
movieId,
1,16
2,14
3,23
4,24
5,34


,release_year
movieId,
1,1995.0
2,1995.0
3,1995.0
4,1995.0
5,1995.0


movieId
1         [Adventure, Animation, Children, Comedy, Fantasy]
2                            [Adventure, Children, Fantasy]
3                                         [Comedy, Romance]
4                                  [Comedy, Drama, Romance]
5                                                  [Comedy]
                                ...                        
161582                                       [Crime, Drama]
161594    [Action, Adventure, Animation, Drama, Fantasy,...
161918                  [Action, Adventure, Horror, Sci-Fi]
163056                 [Action, Adventure, Fantasy, Sci-Fi]
163949                                        [Documentary]
Name: genres, Length: 8737, dtype: object

# Build a content-based model
When ready, move the following class in the *models.py* script

In [3]:
class ContentBased(AlgoBase):
    def __init__(self, features_method, regressor_method):
        AlgoBase.__init__(self)
        self.regressor_method = regressor_method
        self.content_features = self.create_content_features(features_method)

    def create_content_features(self, features_method):
        """Content Analyzer"""
        df_items = load_items()

        if features_method is None:
            df_features = None

        elif features_method == "title_length":
            # Naive feature based on the number of characters in the movie title
            df_features = df_items[C.LABEL_COL].apply(
                lambda x: len(x)
            ).to_frame("n_character_title")

            # Important: convert to float before normalization
            df_features = df_features.astype(float)

            scaler = MinMaxScaler()
            df_features.loc[:, :] = scaler.fit_transform(df_features)

        elif features_method == "date":
            # Extract release year from the movie title
            df_features = df_items[C.LABEL_COL].str.extract(
                r"\((\d{4})\)"
            ).astype(float)

            df_features.columns = ["release_year"]

            # Fill missing years with the median year
            df_features["release_year"] = df_features["release_year"].fillna(
                df_features["release_year"].median()
            )

            # Important: convert to float before normalization
            df_features = df_features.astype(float)

            scaler = MinMaxScaler()
            df_features.loc[:, :] = scaler.fit_transform(df_features)

        else:
            raise NotImplementedError(
                f"Feature method {features_method} not yet implemented"
            )

        return df_features

    def prepare_user_data(self, u):
        """
        Prepare the feature matrix X and ratings y for one user.
        """

        ratings_list = self.trainset.ur[u]

        df_user = pd.DataFrame(
            ratings_list,
            columns=["item_id", "user_ratings"]
        )

        df_user["item_id"] = df_user["item_id"].map(
            self.trainset.to_raw_iid
        )

        df_user["item_id"] = df_user["item_id"].astype(
            self.content_features.index.dtype
        )

        df_user = df_user.merge(
            self.content_features,
            how="left",
            left_on="item_id",
            right_index=True
        )

        df_user = df_user.fillna(0)

        X = df_user[self.content_features.columns].values
        y = df_user["user_ratings"].values

        return X, y

    def compute_user_explanation(self, u):
        """
        Compute a feature importance profile for one user.

        The importance of each feature is computed as the average
        feature value of the movies rated by the user, weighted by the user's ratings.
        """

        if self.content_features is None:
            return {}

        feature_names = self.content_features.columns.tolist()

        X, y = self.prepare_user_data(u)

        if len(y) == 0:
            return {feature: 0 for feature in feature_names}

        # Ratings are used as weights.
        # We make them strictly positive to avoid problems.
        weights = y - np.min(y) + 1e-8

        weighted_features = np.average(
            X,
            axis=0,
            weights=weights
        )

        min_value = np.min(weighted_features)
        max_value = np.max(weighted_features)

        if max_value > min_value:
            normalized_scores = (
                weighted_features - min_value
            ) / (max_value - min_value)
        else:
            normalized_scores = weighted_features

        explanation = {
            feature_names[i]: float(normalized_scores[i])
            for i in range(len(feature_names))
        }

        return explanation

    def fit(self, trainset):
        """Profile Learner"""
        AlgoBase.fit(self, trainset)

        # Preallocate user profiles
        self.user_profile = {
            u: None for u in trainset.all_users()
        }

        # Post-hackathon explanation dictionary
        self.user_profile_explain = {
            u: None for u in trainset.all_users()
        }

        if self.regressor_method == "random_score":
            for u in self.user_profile:
                self.user_profile_explain[u] = self.compute_user_explanation(u)

        elif self.regressor_method == "random_sample":
            for u in self.user_profile:
                self.user_profile[u] = [
                    rating for _, rating in self.trainset.ur[u]
                ]

                self.user_profile_explain[u] = self.compute_user_explanation(u)

        else:
            pass
            # Implement here other regressors if needed

        return self

    def estimate(self, u, i):
        """Scoring component used for item filtering"""

        if not (
            self.trainset.knows_user(u)
            and self.trainset.knows_item(i)
        ):
            raise PredictionImpossible(
                "User and/or item is unknown."
            )

        if self.regressor_method == "random_score":
            rd.seed()
            score = rd.uniform(0.5, 5)

        elif self.regressor_method == "random_sample":
            rd.seed()
            score = rd.choice(self.user_profile[u])

        else:
            raise PredictionImpossible(
                "Regressor method is not implemented."
            )

        return score

    def explain(self, u, top_n=10):
        """
        Return the top_n most important features for a given user.
        """

        try:
            inner_u = self.trainset.to_inner_uid(u)
        except ValueError:
            inner_u = u

        if inner_u not in self.user_profile_explain:
            raise ValueError(f"User {u} not found in the trained model.")

        explanation = self.user_profile_explain[inner_u]

        if explanation is None:
            raise ValueError(f"No explanation available for user {u}.")

        sorted_explanation = dict(
            sorted(
                explanation.items(),
                key=lambda item: item[1],
                reverse=True
            )[:top_n]
        )

        return sorted_explanation

The following script test the ContentBased class

In [4]:
def test_contentbased_class(feature_method, regressor_method):
    """Test the ContentBased class.
    Tries to make a prediction on the first (user,item ) tuple of the anti_test_set
    """
    sp_ratings = load_ratings(surprise_format=True)
    train_set = sp_ratings.build_full_trainset()

    content_algo = ContentBased(feature_method, regressor_method)
    content_algo.fit(train_set)

    anti_test_set_first = train_set.build_anti_testset()[0]

    prediction = content_algo.predict(anti_test_set_first[0], anti_test_set_first[1])

    print(prediction)

# (call here the test functions with different regressor methods)

test_contentbased_class(feature_method = None, regressor_method='random_score')

test_contentbased_class(feature_method = None,regressor_method='random_sample')

test_contentbased_class(feature_method = 'title_length',regressor_method='random_sample')

user: 277        item: 3          r_ui = None   est = 2.75   {'was_impossible': False}
user: 277        item: 3          r_ui = None   est = 5.00   {'was_impossible': False}
user: 277        item: 3          r_ui = None   est = 5.00   {'was_impossible': False}


In [6]:
sp_ratings = load_ratings(surprise_format=True)
train_set = sp_ratings.build_full_trainset()

content_algo = ContentBased(
    features_method="title_length",
    regressor_method="random_sample"
)

content_algo.fit(train_set)

first_user = list(content_algo.user_profile.keys())[0]

content_algo.explain(first_user, top_n=10)

{'n_character_title': 0.11020697167753037}

## Post-hackathon: explaining predictions

The objective of the post-hackathon part is to make the content-based recommender model more interpretable.

For each user, we compute a feature importance profile. This profile is obtained by taking a weighted average of the movie features, using the ratings given by the user as weights.

This means that features associated with highly rated movies receive a higher importance score.

The result is stored in `self.user_profile_explain`, which maps each user to a dictionary of feature scores.

The method `.explain(u)` returns the most important features for a given user.

In this example, the model uses the `date` feature. Therefore, the explanation returns `release_year`, which represents the normalized release year of the movies.

The returned score is approximately 0.780. This score represents the importance of the release year for the selected user, computed from the movies rated by this user.

Since only one feature is used in this example, the explanation contains only one element. With richer features such as genres, genome scores or visual features, the explanation would provide a more detailed description of the user's preferences.

Also, the model uses one content feature: `n_character_title`, which represents the normalized number of characters in the movie title.

The returned score is approximately 0.110. This value represents the importance of the title length for the selected user. It is computed as a weighted average of the title length of the movies rated by this user, using the user's ratings as weights.

Since only one feature is used in this example, the explanation contains only one element. However, this confirms that the explanation mechanism works correctly with the `title_length` feature.